In [ ]:
# !pip -q install "transformers==4.57.1" "accelerate>=0.34" safetensors
import torch, transformers, sys, platform
print('torch', torch.__version__, '| transformers', transformers.__version__, '| python', sys.version.split()[0])
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ---- Configuration (do not change after the first trial starts) ----
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/praxis_appendix_k'      # all checkpoints, logs and results go here
MODEL_ID = 'microsoft/trocr-base-handwritten'
REVISION = 'eaacaf452b06415df8f10bb6fad3a4c11e609406'
LRS = [1e-5, 5e-5]
IMAGE_MODES = ['unchanged', 'crop_pad']
BATCH, ACCUM, MAX_EPOCHS, WARMUP, WD, PATIENCE = 2, 8, 12, 0.05, 0.01, 3
EVAL_BATCH, MAX_NEW_TOKENS, MAX_LABEL_TOKENS = 16, 128, 64
SEED = 42
TARGET = 0.80
import os; os.makedirs(WORK, exist_ok=True)

In [ ]:
# ---- Appendix K utilities: normalization, scoring, intervals, image handling, manifests ----
import csv, hashlib, json, math, os, re, unicodedata, zipfile
from pathlib import Path
import numpy as np
from PIL import Image


def normalize(s):
    """Primary scorer (Section 4.3): NFC, collapse whitespace, case-fold; punctuation and numerals kept."""
    s = unicodedata.normalize('NFC', s or '')
    s = re.sub(r'\s+', ' ', s).strip()
    return s.casefold()


def edit_counts(ref, hyp):
    """Deterministic Levenshtein alignment -> substitutions, deletions, insertions, hits, N."""
    n, m = len(ref), len(hyp)
    d = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        d[i][0] = i
    for j in range(1, m + 1):
        d[0][j] = j
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            c = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i][j] = min(d[i - 1][j - 1] + c, d[i - 1][j] + 1, d[i][j - 1] + 1)
    S = D = I = H = 0
    i, j = n, m
    while i > 0 or j > 0:   # fixed tie-break order: match/sub, deletion, insertion
        if i > 0 and j > 0 and d[i][j] == d[i - 1][j - 1] + (0 if ref[i - 1] == hyp[j - 1] else 1):
            if ref[i - 1] == hyp[j - 1]:
                H += 1
            else:
                S += 1
            i, j = i - 1, j - 1
        elif i > 0 and d[i][j] == d[i - 1][j] + 1:
            D += 1; i -= 1
        else:
            I += 1; j -= 1
    return {'S': S, 'D': D, 'I': I, 'H': H, 'N': n}


def score(refs, hyps):
    """Micro CER/WER (pooled), complete-label exact agreement. Empty outputs stay in the denominator."""
    assert len(refs) == len(hyps)
    ch = {'S': 0, 'D': 0, 'I': 0, 'H': 0, 'N': 0}
    wd = {'S': 0, 'D': 0, 'I': 0, 'H': 0, 'N': 0}
    exact = 0
    for r, h in zip(refs, hyps):
        r, h = normalize(r), normalize(h)
        exact += int(r == h)
        for k, v in edit_counts(list(r), list(h)).items():
            ch[k] += v
        for k, v in edit_counts(r.split(' ') if r else [], h.split(' ') if h else []).items():
            wd[k] += v
    n = len(refs)
    return {
        'n': n, 'exact_count': exact, 'exact': exact / n if n else None,
        'cer': (ch['S'] + ch['D'] + ch['I']) / ch['N'] if ch['N'] else None,
        'wer': (wd['S'] + wd['D'] + wd['I']) / wd['N'] if wd['N'] else None,
        'char_counts': ch, 'word_counts': wd,
    }


def wilson(k, n, z=1.959963984540054):
    if n == 0:
        return (None, None)
    p = k / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (max(0.0, c - h), min(1.0, c + h))


def otsu(gray):
    hist = np.bincount(gray.ravel(), minlength=256).astype(float)
    total = gray.size
    sum_all = np.dot(np.arange(256), hist)
    wb = sb = 0.0
    best, thr = -1.0, 127
    for t in range(256):
        wb += hist[t]
        if wb == 0:
            continue
        wf = total - wb
        if wf == 0:
            break
        sb += t * hist[t]
        mb, mf = sb / wb, (sum_all - sb) / wf
        between = wb * wf * (mb - mf) ** 2
        if between > best:
            best, thr = between, t
    return thr


def crop_and_pad(img, margin=8):
    """Appendix K 'aspect-preserving crop-and-pad': crop to the ink bounding box (Otsu), pad to a white square."""
    rgb = img.convert('RGB')
    g = np.asarray(rgb.convert('L'))
    thr = min(otsu(g), float(np.median(g)) - 40)   # guard against splitting background noise
    ink = g <= thr
    rows = np.where(ink.sum(axis=1) >= 2)[0]
    cols = np.where(ink.sum(axis=0) >= 2)[0]
    if ink.sum() < 10 or len(rows) == 0 or len(cols) == 0:
        return rgb
    x0, x1 = max(0, cols.min() - margin), min(g.shape[1], cols.max() + 1 + margin)
    y0, y1 = max(0, rows.min() - margin), min(g.shape[0], rows.max() + 1 + margin)
    c = rgb.crop((x0, y0, x1, y1))
    side = max(c.size)
    canvas = Image.new('RGB', (side, side), (255, 255, 255))
    canvas.paste(c, ((side - c.size[0]) // 2, (side - c.size[1]) // 2))
    return canvas


def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def is_scorable(ref):
    return not re.fullmatch(r'[?\s]+', ref or '')


def find_split_csv(root, split):
    hits = [p for p in Path(root).rglob('*.csv')
            if re.sub(r'[^a-z]', '', p.stem.lower()) in {split + 'label', split + 'labels'}]
    if len(hits) != 1:
        raise ValueError(f'Expected one {split} label CSV, found {len(hits)}: {hits}')
    return hits[0]


def read_rows(csv_path):
    with open(csv_path, encoding='utf-8-sig', newline='') as f:
        rows = list(csv.DictReader(f))
    cols = {c.lower().strip(): c for c in rows[0].keys()}
    ic = next(cols[c] for c in ('images', 'image', 'image_name', 'filename', 'file') if c in cols)
    tc = next(cols[c] for c in ('text', 'label', 'transcription') if c in cols)
    return [(r[ic].strip(), r[tc].strip()) for r in rows]


def index_images(root):
    idx = {}
    for p in Path(root).rglob('*'):
        if p.suffix.lower() in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}:
            idx.setdefault(p.name, []).append(p)
    return idx


def resolve(rows, idx):
    out = []
    for name, ref in rows:
        c = idx.get(Path(name.replace('\\', '/')).name, [])
        if len(c) != 1:
            raise ValueError(f'Missing or ambiguous image: {name} ({len(c)} matches)')
        out.append({'record_id': c[0].name, 'image_path': str(c[0]), 'reference_text': ref})
    return out


def dhash(img, size=8):
    g = img.convert('L').resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).ravel()
    return int(''.join('1' if b else '0' for b in bits), 2)


def rebuild_split(train_records, exclude_ids=(), seed=42, n_val=669, n_hold=669, max_dist=2):
    """Fallback only: regroup by exact hash + dHash<=2, assign whole groups. NOT the prepared Appendix K split."""
    import random
    hashes = [sha256_file(r['image_path']) for r in train_records]
    dh = [dhash(Image.open(r['image_path'])) for r in train_records]
    parent = list(range(len(train_records)))
    def find(a):
        while parent[a] != a:
            parent[a] = parent[parent[a]]; a = parent[a]
        return a
    by_hash = {}
    for i, h in enumerate(hashes):
        by_hash.setdefault(h, []).append(i)
    for g in by_hash.values():
        for j in g[1:]:
            parent[find(j)] = find(g[0])
    for i in range(len(dh)):
        for j in range(i + 1, len(dh)):
            if bin(dh[i] ^ dh[j]).count('1') <= max_dist:
                parent[find(j)] = find(i)
    groups = {}
    for i in range(len(train_records)):
        groups.setdefault(find(i), []).append(i)
    glist = list(groups.values())
    random.Random(seed).shuffle(glist)
    split = {}
    hold, val = [], []
    for g in glist:
        ids = {train_records[i]['record_id'] for i in g}
        if len(hold) < n_hold and not (ids & set(exclude_ids)) and len(hold) + len(g) <= n_hold:
            hold += g
        elif len(val) < n_val and len(val) + len(g) <= n_val:
            val += g
        else:
            for i in g:
                split[i] = 'train'
            continue
        for i in g:
            split[i] = 'holdout' if i in hold else 'val'
    return {train_records[i]['record_id']: s for i, s in split.items()}

In [ ]:
# ---- Data: unpack RxHandBD and load (or rebuild) the Appendix K partition ----
import glob, shutil
from google.colab import files

DATA = '/content/rxhandbd'
RX_URL = 'https://data.mendeley.com/public-files/datasets/dsb5r6vskg/files/98acd360-ba4c-45d0-b614-5d66f4deee47/file_downloaded'
RX_SHA = 'c6371c0a3a89301c0b6b2eac1d2d49f6f3d8e559a7a71b338655dde0a6a63b51'   # Appendix H; RxHandBD-ML.zip, Version 3
archive = f'{WORK}/RxHandBD-ML.zip'
if not os.path.exists(archive) or sha256_file(archive) != RX_SHA:
    import urllib.request
    print('Downloading RxHandBD-ML.zip (128 MB) from Mendeley Data ...')
    urllib.request.urlretrieve(RX_URL, archive)
ARCHIVE_SHA = sha256_file(archive)
assert ARCHIVE_SHA == RX_SHA, f'Archive hash {ARCHIVE_SHA} differs from Appendix H'
print('archive verified', ARCHIVE_SHA)
if not os.path.exists(DATA):
    with zipfile.ZipFile(archive) as z:
        z.extractall(DATA)

idx = index_images(DATA)
train_all = resolve(read_rows(find_split_csv(DATA, 'train')), idx)
test_all = resolve(read_rows(find_split_csv(DATA, 'test')), idx)
print(len(train_all), 'training rows;', len(test_all), 'test rows')   # expect 4,463 and 1,115

# Prepared Appendix K manifests: put k_train.csv, k_val.csv, k_holdout.csv (any CSV with an image-name column)
# in WORK/manifests. If they are absent, the split is rebuilt with the documented grouping rule, which yields a
# DIFFERENT holdout from the prepared one; that is recorded in the run manifest and must be stated in the praxis.
MAN = f'{WORK}/manifests'
os.makedirs(MAN, exist_ok=True)
by_id = {r['record_id']: r for r in train_all}
paths = {k: f'{MAN}/k_{k}.csv' for k in ('train', 'val', 'holdout')}
if all(os.path.exists(p) for p in paths.values()):
    SPLIT_SOURCE = 'prepared Appendix K manifests'
    part = {}
    for k, p in paths.items():
        for name, _ in read_rows(p):
            part[Path(name).name] = k
else:
    SPLIT_SOURCE = 'REBUILT (prepared manifests not supplied)'
    pilot = f'{MAN}/pilot_120.txt'    # optional list of the 120 preprocessing-pilot image names
    excl = open(pilot).read().split() if os.path.exists(pilot) else []
    part = rebuild_split(train_all, exclude_ids=excl, seed=SEED)
    for k in ('train', 'val', 'holdout'):
        with open(paths[k], 'w', newline='') as f:
            w = csv.writer(f); w.writerow(['Images', 'Text'])
            for rid, s in part.items():
                if s == k:
                    w.writerow([rid, by_id[rid]['reference_text']])
    if not excl:
        print('WARNING: pilot exclusion list not supplied; holdout may include pilot images.')

splits = {k: [by_id[r] for r, s in part.items() if s == k] for k in ('train', 'val', 'holdout')}
for k in splits:
    splits[k] = [r for r in splits[k] if is_scorable(r['reference_text'])]
print(SPLIT_SOURCE, {k: len(v) for k, v in splits.items()})   # expect 3,125 / 669 / 669
assert not (set(r['record_id'] for r in splits['holdout']) & set(r['record_id'] for r in splits['train'] + splits['val']))

In [ ]:
# ---- Model, data loading, evaluation ----
import random, time, math
from torch.utils.data import Dataset, DataLoader
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, get_linear_schedule_with_warmup

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

processor = TrOCRProcessor.from_pretrained(MODEL_ID, revision=REVISION, use_fast=False)
tok = processor.tokenizer

def load_model(path=None):
    m = VisionEncoderDecoderModel.from_pretrained(path or MODEL_ID, revision=None if path else REVISION)
    # Keep the checkpoint's own decoder start token (TrOCR uses </s>, id 2) so the pretrained baseline
    # behaves as in Chapter 4; only fill ids that training needs if they are missing.
    if m.config.decoder_start_token_id is None:
        m.config.decoder_start_token_id = m.generation_config.decoder_start_token_id or tok.eos_token_id
    if m.config.pad_token_id is None:
        m.config.pad_token_id = tok.pad_token_id
    return m.cuda()

def prep(img_path, mode):
    im = Image.open(img_path).convert('RGB')
    return crop_and_pad(im) if mode == 'crop_pad' else im

class Words(Dataset):
    def __init__(self, recs, mode): self.recs, self.mode = recs, mode
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        px = processor(images=prep(r['image_path'], self.mode), return_tensors='pt').pixel_values[0]
        lab = tok(r['reference_text'], max_length=MAX_LABEL_TOKENS, truncation=True).input_ids
        return px, lab

def collate(batch):
    px = torch.stack([b[0] for b in batch])
    L = max(len(b[1]) for b in batch)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    for i, (_, l) in enumerate(batch):
        lab[i, :len(l)] = torch.tensor(l)
    return px, lab

@torch.no_grad()
def predict(model, recs, mode, beams=1):
    model.eval(); out = []
    for i in range(0, len(recs), EVAL_BATCH):
        chunk = recs[i:i + EVAL_BATCH]
        px = processor(images=[prep(r['image_path'], mode) for r in chunk], return_tensors='pt').pixel_values.cuda()
        with torch.autocast('cuda', dtype=torch.float16):
            ids = model.generate(px, max_new_tokens=MAX_NEW_TOKENS, num_beams=beams, do_sample=False,
                                 early_stopping=beams > 1)
        out += tok.batch_decode(ids, skip_special_tokens=True)
    return out

def evaluate(model, recs, mode, beams=1):
    hyps = predict(model, recs, mode, beams)
    return score([r['reference_text'] for r in recs], hyps), hyps

def better(a, b):
    """Selection rule: higher exact agreement; WER breaks ties."""
    if b is None: return True
    return (a['exact'], -a['wer']) > (b['exact'], -b['wer'])

In [ ]:
# ---- Four trials; resumable after a disconnect (state saved each epoch) ----
def run_trial(lr, mode):
    name = f'lr{lr:g}_{mode}'
    tdir = f'{WORK}/trials/{name}'; os.makedirs(tdir, exist_ok=True)
    status_p = f'{tdir}/status.json'
    st = json.load(open(status_p)) if os.path.exists(status_p) else {'epoch': 0, 'history': [], 'best': None, 'done': False}
    if st['done']:
        print(name, 'already complete; best', st['best']); return st
    set_seed(SEED)
    model = load_model(f'{tdir}/latest' if st['epoch'] else None)
    dl = DataLoader(Words(splits['train'], mode), batch_size=BATCH, shuffle=True, collate_fn=collate,
                    num_workers=2, generator=torch.Generator().manual_seed(SEED))
    steps_per_epoch = math.ceil(len(dl) / ACCUM)
    total = steps_per_epoch * MAX_EPOCHS
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WD)
    sch = get_linear_schedule_with_warmup(opt, int(WARMUP * total), total)
    scaler = torch.amp.GradScaler('cuda')
    if st['epoch']:
        ck = torch.load(f'{tdir}/latest/train_state.pt', map_location='cuda')
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); scaler.load_state_dict(ck['scaler'])
    stale = st.get('stale', 0)
    for epoch in range(st['epoch'] + 1, MAX_EPOCHS + 1):
        model.train(); t0 = time.time(); run_loss = 0.0
        opt.zero_grad(set_to_none=True)
        for step, (px, lab) in enumerate(dl, 1):
            with torch.autocast('cuda', dtype=torch.float16):
                loss = model(pixel_values=px.cuda(), labels=lab.cuda()).loss / ACCUM
            scaler.scale(loss).backward(); run_loss += loss.item()
            if step % ACCUM == 0 or step == len(dl):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True)
        val, _ = evaluate(model, splits['val'], mode, beams=1)
        improved = better(val, st['best'])
        rec = {'epoch': epoch, 'train_loss': run_loss / steps_per_epoch, 'val_exact': val['exact'],
               'val_cer': val['cer'], 'val_wer': val['wer'], 'improved': improved, 'minutes': (time.time() - t0) / 60}
        st['history'].append(rec); print(name, rec)
        if improved:
            st['best'] = {k: val[k] for k in ('exact', 'exact_count', 'cer', 'wer')} | {'epoch': epoch}
            model.save_pretrained(f'{tdir}/best'); stale = 0
        else:
            stale += 1
        model.save_pretrained(f'{tdir}/latest')
        torch.save({'opt': opt.state_dict(), 'sch': sch.state_dict(), 'scaler': scaler.state_dict()},
                   f'{tdir}/latest/train_state.pt')
        st['epoch'] = epoch; st['stale'] = stale
        if stale >= PATIENCE or epoch == MAX_EPOCHS:
            st['done'] = True; st['stopped_early'] = stale >= PATIENCE
        json.dump(st, open(status_p, 'w'), indent=1)
        if st['done']:
            break
    shutil.rmtree(f'{tdir}/latest', ignore_errors=True)   # keep only the best weights per trial
    del model; torch.cuda.empty_cache()
    return st

results = {}
for lr in LRS:
    for mode in IMAGE_MODES:
        results[(lr, mode)] = run_trial(lr, mode)

In [ ]:
# ---- Selection on validation only: trial, then decoding (greedy vs. beam 4); freeze and hash ----
best_key, best = None, None
for k, st in results.items():
    if better(st['best'], best):
        best_key, best = k, st['best']
lr, mode = best_key
sel_dir = f'{WORK}/trials/lr{lr:g}_{mode}/best'
model = load_model(sel_dir)
dec = {}
for beams in (1, 4):
    dec[beams], _ = evaluate(model, splits['val'], mode, beams)
    print('validation beams', beams, {k: dec[beams][k] for k in ('exact', 'cer', 'wer')})
BEAMS = 4 if better(dec[4], dec[1]) and dec[4] != dec[1] else 1

frozen = {
    'selected_trial': {'lr': lr, 'image_mode': mode, 'epoch': best['epoch']},
    'decoding': {'num_beams': BEAMS, 'max_new_tokens': MAX_NEW_TOKENS, 'do_sample': False},
    'validation': {'trials': {f'lr{k[0]:g}_{k[1]}': v['best'] for k, v in results.items()},
                   'decoding': {str(b): {x: dec[b][x] for x in ('exact', 'exact_count', 'cer', 'wer')} for b in dec}},
    'checkpoint_sha256': {p.name: sha256_file(p) for p in sorted(Path(sel_dir).glob('*')) if p.is_file()},
    'manifest_sha256': {k: sha256_file(p) for k, p in paths.items()},
    'split_source': SPLIT_SOURCE,
    'protocol': {'init': MODEL_ID, 'revision': REVISION, 'lrs': LRS, 'image_modes': IMAGE_MODES, 'batch': BATCH,
                 'accum': ACCUM, 'max_epochs': MAX_EPOCHS, 'warmup': WARMUP, 'weight_decay': WD,
                 'patience': PATIENCE, 'seed': SEED, 'optimizer': 'AdamW', 'fp16': True, 'grad_clip': 1.0},
    'environment': {'torch': torch.__version__, 'transformers': transformers.__version__,
                    'gpu': torch.cuda.get_device_name(0), 'frozen_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')},
}
FROZEN = f'{WORK}/frozen_selection.json'
if os.path.exists(FROZEN):
    # A previous session already froze the selection: never re-select (the holdout may have been opened).
    frozen = json.load(open(FROZEN))
    print('Using the selection frozen earlier.')
else:
    json.dump(frozen, open(FROZEN, 'w'), indent=1)
print('FROZEN:', frozen['selected_trial'], frozen['decoding'], '| sha256', sha256_file(FROZEN))

In [ ]:
# ---- Holdout: opened once. Fine-tuned model and pretrained baseline, same images, preprocessing, decoding ----
fz = json.load(open(FROZEN))
mode, BEAMS = fz['selected_trial']['image_mode'], fz['decoding']['num_beams']
sel_dir = f"{WORK}/trials/lr{fz['selected_trial']['lr']:g}_{mode}/best"
hold = splits['holdout']
ft_model = load_model(sel_dir)
ft, ft_h = evaluate(ft_model, hold, mode, BEAMS)
del ft_model; torch.cuda.empty_cache()
base_model = load_model()
bl, bl_h = evaluate(base_model, hold, mode, BEAMS)

with open(f'{WORK}/holdout_predictions.jsonl', 'w') as f:
    for r, a, b in zip(hold, ft_h, bl_h):
        f.write(json.dumps({'record_id': r['record_id'], 'reference': r['reference_text'],
                            'finetuned': a, 'pretrained': b}, ensure_ascii=False) + '\n')

def row(s):
    lo, hi = wilson(s['exact_count'], s['n'])
    return {'n': s['n'], 'exact_count': s['exact_count'], 'exact': s['exact'], 'exact_wilson95': [lo, hi],
            'cer': s['cer'], 'wer': s['wer'], 'char_counts': s['char_counts'], 'word_counts': s['word_counts']}
holdout_results = {'finetuned': row(ft), 'pretrained_baseline': row(bl),
                   'target': TARGET, 'target_met': ft['exact'] >= TARGET,
                   'min_correct_for_target': math.ceil(TARGET * len(hold))}
json.dump(holdout_results, open(f'{WORK}/holdout_results.json', 'w'), indent=1)
for k in ('finetuned', 'pretrained_baseline'):
    r = holdout_results[k]
    print(f"{k:20s} exact {r['exact_count']}/{r['n']} = {r['exact']:.2%} "
          f"[{r['exact_wilson95'][0]:.2%}, {r['exact_wilson95'][1]:.2%}]  CER {r['cer']:.2%}  WER {r['wer']:.2%}")
print('80% target met:', holdout_results['target_met'])

In [ ]:
# ---- Secondary (clearly labeled): fine-tuned model on the original 1,115-image test partition ----
# Comparable to Table 4-2 (pretrained 9.87%). Secondary because the original test errors were inspected earlier.
test_primary = [r for r in test_all if is_scorable(r['reference_text'])]   # 1,114
ft_model = load_model(sel_dir)
sec, sec_h = evaluate(ft_model, test_primary, mode, BEAMS)
secondary = row(sec)
json.dump(secondary, open(f'{WORK}/secondary_original_test.json', 'w'), indent=1)
print(f"original test (secondary): exact {sec['exact_count']}/{sec['n']} = {sec['exact']:.2%}  CER {sec['cer']:.2%}  WER {sec['wer']:.2%}")

In [ ]:
# ---- Package everything for the praxis (send results_for_praxis.zip back) ----
bundle = {'frozen_selection': fz, 'holdout': holdout_results, 'secondary_original_test': secondary,
          'trial_histories': {f'lr{k[0]:g}_{k[1]}': v['history'] for k, v in results.items()}}
json.dump(bundle, open(f'{WORK}/results_for_praxis.json', 'w'), indent=1)
zp = f'{WORK}/results_for_praxis.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in ['results_for_praxis.json', 'frozen_selection.json', 'holdout_results.json',
              'holdout_predictions.jsonl', 'secondary_original_test.json']:
        z.write(f'{WORK}/{p}', p)
    for p in glob.glob(f'{WORK}/manifests/*.csv'):
        z.write(p, 'manifests/' + os.path.basename(p))
files.download(zp)